# DyTopo — Local Evaluation via SAP Hyperspace AI Proxy

Run DyTopo benchmarks from your **local Mac (M3 Pro, 36 GB, no GPU)** using the
SAP Hyperspace AI LLM Proxy as the inference backend.

**No GPU required.** `all-MiniLM-L6-v2` (embedding model) runs locally on CPU/MPS.
LLM inference is handled remotely via the Hyperspace OpenAI-compatible API.

## Pre-requisites
1. SAP Hyperspace AI access — get your endpoint + API key from the internal portal
2. `pip install sentence-transformers datasets httpx nest_asyncio tqdm pandas matplotlib`

## Cost estimate (734 problems, ~2.6 rounds avg)
| Model | Prompt $/1M | Completion $/1M | Est. total |
|---|---|---|---|
| claude-haiku-4-5-20251001 | $0.80 | $4.00 | **~$50** |
| claude-sonnet-4-5 | $3.00 | $15.00 | **~$189** |
| gpt-4o | $2.50 | $10.00 | **~$133** |

In [2]:
# ── Cell 1: Config ────────────────────────────────────────────────────────────
HYPERSPACE_BASE_URL = "http://localhost:6655"   # <-- fill in
HYPERSPACE_API_KEY  = "c8f08eb5-9a37-43a8-a7a7-272502df91e6"               # <-- fill in
HYPERSPACE_MODEL    = "claude-haiku-4-5-20251001"             # or "claude-sonnet-4-5"

BACKEND      = "hyperspace"
LLM_BASE_URL = HYPERSPACE_BASE_URL
LLM_MODEL    = HYPERSPACE_MODEL
LLM_API_KEY  = HYPERSPACE_API_KEY
MODEL_LABEL  = f"Hyperspace / {LLM_MODEL}"

from pathlib import Path
RESULTS_DIR = Path('./results_hyperspace')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Backend : {BACKEND}")
print(f"Model   : {LLM_MODEL}")
print(f"Endpoint: {LLM_BASE_URL}")
print(f"Results : {RESULTS_DIR}")

Backend : hyperspace
Model   : claude-haiku-4-5-20251001
Endpoint: http://localhost:6655
Results : results_hyperspace


In [3]:
import urllib.request
src = urllib.request.urlopen(
    'https://raw.githubusercontent.com/51103220/agentic-dynamic-topology/refs/heads/main/api/dytopo_engine.py'
).read().decode()
checks = [
    ('Manager final_answer field',       '"final_answer": "String. REQUIRED when is_complete=true' in src),
    ('Manager final_answer extraction',  'mgr_raw.get("final_answer", "").strip()' in src),
    ('best_primary_answer tracking',     'best_primary_answer' in src),
    ('llm_api_key field',                'llm_api_key' in src),
    ('tokens_in/out in RoundResult',     'tokens_in' in src),
    ('Aggregation order σ(t)',           'build_aggregation_order' in src),
    ('K_in cap applied',                 'apply_k_in_cap' in src),
]
all_ok = True
for name, ok in checks:
    print(f'  {"✓" if ok else "✗ MISSING"}  {name}')
    if not ok: all_ok = False
print()
print('Engine READY' if all_ok else 'ENGINE NOT UPDATED — commit & push first')

  ✓  Manager final_answer field
  ✓  Manager final_answer extraction
  ✓  best_primary_answer tracking
  ✗ MISSING  llm_api_key field
  ✗ MISSING  tokens_in/out in RoundResult
  ✓  Aggregation order σ(t)
  ✓  K_in cap applied

ENGINE NOT UPDATED — commit & push first


In [ ]:
import urllib.request, os, sys
from pathlib import Path

ENGINE_URL = 'https://raw.githubusercontent.com/51103220/agentic-dynamic-topology/refs/heads/main/api/dytopo_engine.py'
ENGINE_PATH = Path('/tmp/dytopo_engine.py') if RESULTS_DIR.parts[1] != 'workspace' else Path('/workspace/dytopo_engine.py')
ENGINE_PATH.parent.mkdir(parents=True, exist_ok=True)

urllib.request.urlretrieve(ENGINE_URL, ENGINE_PATH)
print(f'Downloaded dytopo_engine.py ({ENGINE_PATH.stat().st_size:,} bytes)')

if str(ENGINE_PATH.parent) not in sys.path:
    sys.path.insert(0, str(ENGINE_PATH.parent))

import nest_asyncio; nest_asyncio.apply()
from dytopo_engine import DyTopoConfig, run_dytopo, encode, build_adjacency
print('dytopo_engine imported')

In [ ]:
print('Loading datasets...')
from datasets import load_dataset
import random, json, re, subprocess, sys
random.seed(42)

humaneval_ds = load_dataset('openai/openai_humaneval', split='test')
math500_ds   = load_dataset('HuggingFaceH4/MATH-500',  split='test')
omnimath_ds  = load_dataset('KbsdJames/Omni-MATH',     split='test')
print(f'HumanEval {len(humaneval_ds)} | MATH-500 {len(math500_ds)} | Omni-MATH {len(omnimath_ds)}')

humaneval_problems = [
    {'id': x['task_id'], 'problem': x['prompt'], 'canonical': x['canonical_solution'],
     'test': x['test'], 'entry_point': x['entry_point'], 'dataset': 'humaneval'}
    for x in humaneval_ds
]
math500_problems = [
    {'id': f'math500_{i}', 'problem': x['problem'], 'answer': x['answer'],
     'subject': x['subject'], 'level': x['level'], 'dataset': 'math500'}
    for i, x in enumerate(math500_ds)
]

def sample_omnimath(ds, n_domains=7, n_per_domain=10, seed=42):
    rng = random.Random(seed)
    dom_key = 'domain' if 'domain' in ds.column_names else 'subject'
    by_dom = {}
    for i, x in enumerate(ds):
        raw = x.get(dom_key, 'unknown')
        d = (raw[0] if isinstance(raw, list) and raw else str(raw)).split('/')[0].strip()
        by_dom.setdefault(d, []).append((i, x))
    top = sorted(by_dom, key=lambda k: -len(by_dom[k]))[:n_domains]
    out = []
    for dom in top:
        for idx, x in rng.sample(by_dom[dom], min(n_per_domain, len(by_dom[dom]))):
            out.append({'id': f'omnimath_{idx}',
                        'problem': x.get('problem', x.get('question', '')),
                        'answer': x.get('answer', x.get('solution', '')),
                        'domain': dom, 'dataset': 'omnimath'})
    return out

omnimath_problems = sample_omnimath(omnimath_ds)
apps_problems = []
print(f'Problems: HumanEval={len(humaneval_problems)} MATH-500={len(math500_problems)} Omni-MATH={len(omnimath_problems)}')

In [ ]:
import re, subprocess, sys

def extract_code(text):
    m = re.search(r'''```python\s*\n(.+?)```''', text, re.DOTALL)
    if m: return m.group(1)
    m = re.search(r'''```\s*\n(.+?)```''', text, re.DOTALL)
    if m: return m.group(1)
    return text

def eval_humaneval(code, problem, timeout=5.0):
    script = f'{code}\n\n{problem["test"]}\n\ncheck({problem["entry_point"]})\nprint("PASS")'
    try:
        r = subprocess.run([sys.executable, '-c', script], capture_output=True, text=True, timeout=timeout)
        return {'passed': r.returncode == 0 and 'PASS' in r.stdout,
                'error': r.stderr.strip()[:200] if r.returncode != 0 else None}
    except subprocess.TimeoutExpired:
        return {'passed': False, 'error': f'Timeout {timeout}s'}

def normalize_math(s):
    if not s: return ''
    m = re.search(r'''\\boxed\{(.+?)\}''', s, re.DOTALL)
    if m: s = m.group(1)
    else:
        m = re.search(r'[Aa]nswer\s*:\s*(.+?)(?:\n|$)', s)
        if m: s = m.group(1)
    s = re.sub(r'''\\frac\{([^}]+)\}\{([^}]+)\}''', r'\1/\2', s)
    s = s.strip().strip('$').strip().rstrip('.')
    return re.sub(r'\s+', '', s).lower()

def eval_math(output, ref):
    mn, rn = normalize_math(output), normalize_math(ref)
    passed = mn == rn
    if not passed:
        nums_m = re.findall(r'-?\d+(?:[./]\d+)?', mn)
        nums_r = re.findall(r'-?\d+(?:[./]\d+)?', rn)
        if nums_m and nums_r and nums_m[-1] == nums_r[-1]:
            passed = True
    return {'passed': passed, 'model_answer': mn, 'ref_answer': rn}

print('Eval utilities ready')

In [ ]:
import asyncio, json, time
from pathlib import Path
from tqdm.notebook import tqdm

async def run_full_dataset(problems, task_type, results_path,
                           tau=0.3, max_rounds=5, concurrency=2, problem_timeout=180.0):
    all_results = []
    done_ids = set()
    if results_path.exists():
        with open(results_path) as f:
            all_results = json.load(f)
        done_ids = {r['id'] for r in all_results}
        print(f'  Resuming: {len(done_ids)}/{len(problems)} already done')

    cfg = DyTopoConfig(
        task_type=task_type, n_agents=4, tau=tau, max_rounds=max_rounds,
        mock_mode=False,
        llm_base_url=LLM_BASE_URL, llm_model=LLM_MODEL, llm_api_key=LLM_API_KEY,
        temperature=0.3, max_tokens=1200,
    )
    remaining = [p for p in problems if p['id'] not in done_ids]
    sem = asyncio.Semaphore(concurrency)

    async def run_one(prob):
        async with sem:
            t0 = time.monotonic()
            try:
                rounds, tok_in, tok_out = [], 0, 0
                async def _collect():
                    nonlocal tok_in, tok_out
                    async for rr in run_dytopo(prob['problem'], cfg):
                        d = rr.to_dict(); rounds.append(d)
                        tok_in  += d.get('tokens_in',  0)
                        tok_out += d.get('tokens_out', 0)
                await asyncio.wait_for(_collect(), timeout=problem_timeout)
                final = next((r['final_answer'] for r in reversed(rounds) if r.get('final_answer')), '')
                return {
                    'id': prob['id'], 'dataset': prob['dataset'],
                    'n_rounds': len(rounds), 'final_answer': final,
                    'time_s': round(time.monotonic()-t0, 2),
                    'adjacency_per_round': [r['adjacency'] for r in rounds],
                    'tokens_in': tok_in, 'tokens_out': tok_out,
                    'tokens_total': tok_in + tok_out,
                    'model': LLM_MODEL, 'backend': BACKEND,
                }
            except asyncio.TimeoutError:
                return {'id': prob['id'], 'dataset': prob['dataset'],
                        'error': f'Timeout after {problem_timeout}s',
                        'time_s': round(time.monotonic()-t0, 2)}
            except Exception as e:
                return {'id': prob['id'], 'dataset': prob['dataset'],
                        'error': f'{type(e).__name__}: {e}',
                        'time_s': round(time.monotonic()-t0, 2)}

    pbar = tqdm(total=len(remaining), desc='problems', unit='prob')
    errors, t_start = 0, time.monotonic()
    for i in range(0, len(remaining), 1):
        r = (await asyncio.gather(run_one(remaining[i])))[0]
        all_results.append(r)
        if 'error' in r:
            errors += 1; pbar.write(f' error {r["id"]}  {r["error"]}')
        else:
            pbar.write(f' ok {r["id"]}  {r["time_s"]:.0f}s  rounds={r["n_rounds"]}  tok={r.get("tokens_total",0):,}')
        with open(results_path, 'w') as f: json.dump(all_results, f, indent=2)
        done_so_far = len(all_results) - len(done_ids)
        elapsed = time.monotonic() - t_start
        rate = done_so_far / elapsed if elapsed > 0 else 0
        eta_s = (len(remaining)-done_so_far)/rate if rate > 0 else float('inf')
        pbar.update(1)
        pbar.set_postfix(saved=len(all_results), errors=errors,
                         rate=f'{rate:.3f}/s', eta=f'{eta_s/3600:.1f}h' if eta_s < 86400 else '?')
    pbar.close()
    print(f'\n  Done: {len(all_results)} | errors: {sum(1 for r in all_results if "error" in r)} | {results_path}')
    return all_results

print(f'Eval loop ready | backend={BACKEND} | model={LLM_MODEL}')

In [ ]:
DATASETS_TO_RUN = [
    ('humaneval', humaneval_problems, 'code', 0.3, 5),
    ('math500',   math500_problems,   'math', 0.3, 5),
    ('omnimath',  omnimath_problems,  'math', 0.4, 5),
]
all_run_results = {}
for ds_name, problems, task_type, tau, max_rounds in DATASETS_TO_RUN:
    results_path = RESULTS_DIR / f'{ds_name}_results.json'
    print(f'\n═══ {ds_name.upper()} ({len(problems)} problems, τ={tau}) ═══')
    results = await run_full_dataset(problems, task_type, results_path,
                                     tau=tau, max_rounds=max_rounds, concurrency=3)
    all_run_results[ds_name] = results
print('\nAll datasets done.')

In [ ]:
import json
from pathlib import Path

# reload from disk (safe even if running scoring standalone)
all_run_results = {}
for ds_name in ['humaneval', 'math500', 'omnimath']:
    p = RESULTS_DIR / f'{ds_name}_results.json'
    if p.exists():
        with open(p) as f: all_run_results[ds_name] = json.load(f)
        print(f'  {ds_name:12s}: {len(all_run_results[ds_name])} results')

prob_maps = {
    'humaneval': {p['id']: p for p in humaneval_problems},
    'math500':   {p['id']: p for p in math500_problems},
    'omnimath':  {p['id']: p for p in omnimath_problems},
}

def score_dataset(results, ds_name):
    prob_map = prob_maps[ds_name]
    passed = errors = 0
    for r in results:
        if 'error' in r: errors += 1; continue
        prob = prob_map.get(r['id'])
        if not prob: continue
        if ds_name == 'humaneval':
            ok = eval_humaneval(extract_code(r.get('final_answer', '')), prob)['passed']
        else:
            ok = eval_math(r.get('final_answer', ''), prob.get('answer', ''))['passed']
        if ok: passed += 1
    total = len(results)
    return {'dataset': ds_name, 'total': total, 'passed': passed,
            'errors': errors, 'accuracy': passed/total*100 if total else 0}

scores = {}
for ds_name, results in all_run_results.items():
    s = score_dataset(results, ds_name)
    scores[ds_name] = s
    print(f'{ds_name:18s}: {s["passed"]}/{s["total"]} ({s["accuracy"]:.1f}%)  errors={s["errors"]}')

In [ ]:
import matplotlib.pyplot as plt, numpy as np

paper_data = {
    'HumanEval': {'single_agent': 18.29, 'random_topo': 80.49, 'agentscope': 80.00, 'dytopo_paper': 89.63},
    'APPS-Comp': {'single_agent':  9.62, 'random_topo': 24.24, 'agentscope': 23.12, 'dytopo_paper': 25.14},
    'MATH-500':  {'single_agent': 48.57, 'random_topo': 68.57, 'agentscope': 72.86, 'dytopo_paper': 75.71},
    'Omni-MATH': {'single_agent': 21.43, 'random_topo': 25.71, 'agentscope': 35.71, 'dytopo_paper': 51.43},
}
ds_labels = list(paper_data.keys())
ds_map    = {'HumanEval': 'humaneval', 'APPS-Comp': 'apps', 'MATH-500': 'math500', 'Omni-MATH': 'omnimath'}
x = np.arange(len(ds_labels)); width = 0.18
COLORS = {'single_agent':'#475569', 'random_topo':'#f87171', 'agentscope':'#fbbf24',
          'dytopo_paper':'#4ade80', 'dytopo_local':'#38bdf8'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#0a0e1a')

ax = axes[0]; ax.set_facecolor('#0f1828')
for i, m in enumerate(['single_agent','random_topo','agentscope','dytopo_paper']):
    ax.bar(x+(i-1.5)*width, [paper_data[d][m] for d in ds_labels], width,
           label=m.replace('_',' ').title(), color=COLORS[m], alpha=0.85)
local_vals = [scores.get(ds_map[d], {}).get('accuracy') for d in ds_labels]
if any(v is not None for v in local_vals):
    ax.scatter(x, [v or 0 for v in local_vals], marker='*', s=300,
               color=COLORS['dytopo_local'], zorder=5, label=f'DyTopo local ({MODEL_LABEL})')
ax.set_xticks(x); ax.set_xticklabels(ds_labels, color='white', fontsize=9)
ax.set_ylabel('Accuracy (%)', color='white')
ax.set_title('DyTopo vs Baselines', color='white')
ax.legend(facecolor='#1a2235', labelcolor='white', fontsize=8)
ax.tick_params(colors='white'); ax.spines[:].set_color('#1e293b'); ax.grid(axis='y', alpha=0.2)

ax2 = axes[1]; ax2.set_facecolor('#0f1828')
for i, m in enumerate(['random_topo','agentscope','dytopo_paper']):
    ax2.bar(x+(i-1)*width, [paper_data[d][m]-paper_data[d]['single_agent'] for d in ds_labels],
            width, label=m.replace('_',' ').title(), color=COLORS[m], alpha=0.85)
if any(v is not None for v in local_vals):
    ax2.scatter(x, [(scores.get(ds_map[d],{}).get('accuracy') or 0)-paper_data[d]['single_agent']
                    for d in ds_labels], marker='*', s=300, color=COLORS['dytopo_local'],
                zorder=5, label=f'DyTopo local ({MODEL_LABEL})')
ax2.axhline(0, color='#94a3b8', linewidth=1, linestyle='--')
ax2.set_xticks(x); ax2.set_xticklabels(ds_labels, color='white', fontsize=9)
ax2.set_ylabel('Improvement over Single Agent (pp)', color='white')
ax2.set_title('Gain from Multi-Agent Routing', color='white')
ax2.legend(facecolor='#1a2235', labelcolor='white', fontsize=8)
ax2.tick_params(colors='white'); ax2.spines[:].set_color('#1e293b'); ax2.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'results_comparison.png'), dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print(f'Saved to {RESULTS_DIR}/results_comparison.png')

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, numpy as np, json
from pathlib import Path

PRICING = {
    'claude-haiku-4-5-20251001': {'in': 0.80,  'out': 4.00},
    'claude-sonnet-4-5':         {'in': 3.00,  'out': 15.00},
    'claude-opus-5':             {'in': 15.00, 'out': 75.00},
    'gpt-4o':                    {'in': 2.50,  'out': 10.00},
    'gpt-4o-mini':               {'in': 0.15,  'out': 0.60},
    'dytopo-model':              {'in': 0.00,  'out': 0.00},
}

def compute_cost(tok_in, tok_out, model):
    p = PRICING.get(model, {'in': 0.0, 'out': 0.0})
    return (tok_in * p['in'] + tok_out * p['out']) / 1_000_000

rows = []
for ds_name, results in all_run_results.items():
    ok = [r for r in results if 'error' not in r]
    if not ok: continue
    ti = sum(r.get('tokens_in',  0) for r in ok)
    to = sum(r.get('tokens_out', 0) for r in ok)
    tt = sum(r.get('tokens_total', ti+to) for r in ok)
    model = ok[0].get('model', LLM_MODEL)
    rows.append({
        'dataset': ds_name, 'problems': len(ok),
        'tokens_in': ti, 'tokens_out': to, 'tokens_total': tt,
        'avg_tok/problem': round(tt/len(ok)),
        'avg_rounds': round(sum(r.get('n_rounds',0) for r in ok)/len(ok), 2),
        'model': model,
        'est_cost_usd': round(compute_cost(ti, to, model), 2),
    })

df = pd.DataFrame(rows)
totals = {'dataset': 'TOTAL', 'problems': df['problems'].sum(),
          'tokens_in': df['tokens_in'].sum(), 'tokens_out': df['tokens_out'].sum(),
          'tokens_total': df['tokens_total'].sum(),
          'avg_tok/problem': round(df['tokens_total'].sum()/max(df['problems'].sum(),1)),
          'avg_rounds': round(df['avg_rounds'].mean(), 2),
          'model': LLM_MODEL, 'est_cost_usd': round(df['est_cost_usd'].sum(), 2)}
df_disp = pd.concat([df, pd.DataFrame([totals])], ignore_index=True)

print(f'\n{"="*70}')
print(f'  Token & Cost  |  model: {LLM_MODEL}  |  backend: {BACKEND}')
print(f'{"="*70}')
print(df_disp[['dataset','problems','tokens_in','tokens_out','tokens_total',
               'avg_tok/problem','avg_rounds','est_cost_usd']].to_string(index=False))
print(f'\n  paper Table 8 reference: HumanEval DyTopo = 9,453 tok/problem')

he = df[df['dataset']=='humaneval']['avg_tok/problem'].values
if len(he):
    print(f'  HumanEval ours={he[0]:,}  paper=9,453  ratio={he[0]/9453:.2f}x')

# ── Charts ────────────────────────────────────────────────────────────────────
ds_names = df['dataset'].tolist()
x = np.arange(len(ds_names)); w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0e1a')

ax1 = axes[0]; ax1.set_facecolor('#0f1828')
ax1.bar(x-w/2, df['tokens_in']/1e6,  w, label='Prompt (M)',     color='#38bdf8', alpha=0.85)
ax1.bar(x+w/2, df['tokens_out']/1e6, w, label='Completion (M)', color='#f87171', alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(ds_names, color='white')
ax1.set_ylabel('Tokens (millions)', color='white')
ax1.set_title('Token consumption by dataset', color='white')
ax1.legend(facecolor='#1a2235', labelcolor='white', fontsize=9)
ax1.tick_params(colors='white'); ax1.spines[:].set_color('#1e293b'); ax1.grid(axis='y', alpha=0.2)

ax2 = axes[1]; ax2.set_facecolor('#0f1828')
ax2.bar(x, df['avg_tok/problem'], color='#4ade80', alpha=0.85)
if 'humaneval' in ds_names:
    ax2.axhline(9453, color='#fbbf24', linewidth=1.5, linestyle='--', label='Paper Table 8 (9,453)')
ax2.set_xticks(x); ax2.set_xticklabels(ds_names, color='white')
ax2.set_ylabel('Avg tokens / problem', color='white')
ax2.set_title('Token efficiency per problem', color='white')
ax2.legend(facecolor='#1a2235', labelcolor='white', fontsize=9)
ax2.tick_params(colors='white'); ax2.spines[:].set_color('#1e293b'); ax2.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'token_consumption.png'), dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print(f'Saved to {RESULTS_DIR}/token_consumption.png')